In [1]:
import math
import sys

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

ENTITY = 'kejian-zhao-tsinghua-university'
SOURCE_PROJECT = 'work_0320'
TARGET_PROJECT = 'work_0320'

COPY_SPECS = [
    {
        'source_names': ['ft_ir101_s3_full_03-30_0', 'ft_ir101_s3_full_03-31_0'],
        'target_name': 'ft_ir101_s3_full_h-0.333',
    },
]

SUMMARY_KEY_PREFIX = 'summary/work_tpir_at_far_'
SUMMARY_KEY_REPLACEMENT = 'summary/work_0213_tpir_at_far_'
DROP_HISTORY_PREFIXES = ('system/', '_')
STEP_METRIC_CANDIDATES = ('trainer/global_step', 'step', 'epoch')


def project_path(project):
    return f'{ENTITY}/{project}'


def rename_key(key):
    if isinstance(key, str) and key.startswith(SUMMARY_KEY_PREFIX):
        return key.replace(SUMMARY_KEY_PREFIX, SUMMARY_KEY_REPLACEMENT, 1)
    return key


def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, bool) else False


def clean_config(config):
    if not isinstance(config, dict):
        return {}
    return {k: v for k, v in config.items() if not str(k).startswith('_')}


def dedupe_keep_order(items):
    result = []
    seen = set()
    for item in items:
        marker = repr(item)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(item)
    return result


def get_run_by_name(api, project, run_name):
    matches = [run for run in api.runs(project_path(project)) if run.name == run_name]
    if not matches:
        raise ValueError(f'在 {project_path(project)} 中找不到 run: {run_name}')
    if len(matches) > 1:
        raise ValueError(
            f'在 {project_path(project)} 中找到多个同名 run: {run_name}, ids={[run.id for run in matches]}'
        )
    return matches[0]


def target_run_exists(api, run_name):
    return any(run.name == run_name for run in api.runs(project_path(TARGET_PROJECT)))


def fetch_history_df(run):
    rows = list(run.scan_history())
    history_df = pd.DataFrame(rows)
    if history_df.empty:
        return history_df

    drop_cols = [
        col for col in history_df.columns
        if any(col.startswith(prefix) for prefix in DROP_HISTORY_PREFIXES)
    ]
    history_df = history_df.drop(columns=drop_cols, errors='ignore')

    rename_map = {
        col: rename_key(col)
        for col in history_df.columns
        if rename_key(col) != col
    }
    if rename_map:
        history_df = history_df.rename(columns=rename_map)
    return history_df


def fetch_summary_dict(run):
    summary = {}
    for key, value in dict(run.summary).items():
        if str(key).startswith('_'):
            continue
        summary[rename_key(key)] = value
    return summary


def merge_configs(runs):
    merged = {}
    for run in runs:
        merged.update(clean_config(run.config))
    return merged


def merge_tags(runs):
    tags = []
    for run in runs:
        tags.extend(list(run.tags or []))
    tags.append(f'copied-from:{SOURCE_PROJECT}')
    if len(runs) > 1:
        tags.append('merged')
    return dedupe_keep_order(tags)


def merged_group(runs):
    groups = dedupe_keep_order([run.group for run in runs if run.group])
    return groups[0] if len(groups) == 1 else None


def build_notes(runs, target_name):
    lines = [
        f'Copied to {ENTITY}/{TARGET_PROJECT} as {target_name}.',
        f'Source project: {ENTITY}/{SOURCE_PROJECT}.',
    ]
    for run in runs:
        lines.append(f'- {run.name} ({run.id})')
    return '\n'.join(lines)


def find_step_metric(history_df):
    for candidate in STEP_METRIC_CANDIDATES:
        if candidate in history_df.columns and history_df[candidate].notna().any():
            return candidate
    return None


def build_payload(api, spec):
    runs = [get_run_by_name(api, SOURCE_PROJECT, run_name) for run_name in spec['source_names']]
    history_frames = [fetch_history_df(run) for run in runs]
    history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()
    renamed_summary_keys = sorted({
        rename_key(key)
        for run in runs
        for key in dict(run.summary).keys()
        if not str(key).startswith('_') and rename_key(key) != key
    })
    return {
        'runs': runs,
        'target_name': spec['target_name'],
        'history_df': history_df,
        'summary': fetch_summary_dict(runs[-1]),
        'config': merge_configs(runs),
        'tags': merge_tags(runs),
        'group': merged_group(runs),
        'notes': build_notes(runs, spec['target_name']),
        'renamed_summary_keys': renamed_summary_keys,
    }


def preview_payload(payload):
    print('=' * 80)
    print(f"target: {payload['target_name']}")
    print(f"source runs: {[run.name for run in payload['runs']]}")
    print(f"source ids: {[run.id for run in payload['runs']]}")
    print(f"history rows: {len(payload['history_df'])}")
    print(f"history columns: {list(payload['history_df'].columns)}")
    print(f"step metric: {find_step_metric(payload['history_df'])}")
    print(f"summary size: {len(payload['summary'])}")
    print(f"renamed summary keys: {payload['renamed_summary_keys']}")
    print(f"tags: {payload['tags']}")
    print(f"group: {payload['group']}")


def configure_default_step_metric(history_df):
    step_metric = find_step_metric(history_df)
    if step_metric:
        wandb.define_metric(step_metric)
        wandb.define_metric('*', step_metric=step_metric)
    return step_metric


def upload_history(history_df):
    if history_df.empty:
        return
    for _, row in tqdm(history_df.iterrows(), total=len(history_df), desc='uploading'):
        log_dict = {}
        for col, value in row.items():
            if not is_missing_value(value):
                log_dict[col] = value
        if log_dict:
            wandb.log(log_dict)


def create_target_run(api, payload):
    if target_run_exists(api, payload['target_name']):
        raise ValueError(
            f"目标项目 {project_path(TARGET_PROJECT)} 中已经存在 run: {payload['target_name']}"
        )

    init_kwargs = {
        'entity': ENTITY,
        'project': TARGET_PROJECT,
        'name': payload['target_name'],
        'config': payload['config'],
        'tags': payload['tags'],
        'notes': payload['notes'],
    }
    if payload['group']:
        init_kwargs['group'] = payload['group']

    new_run = wandb.init(**init_kwargs)
    print(f"created: {new_run.id} | {new_run.url}")

    step_metric = configure_default_step_metric(payload['history_df'])
    print(f'step metric = {step_metric}')

    upload_history(payload['history_df'])

    for key, value in payload['summary'].items():
        wandb.run.summary[key] = value

    wandb.finish()
    print(f"finished: {payload['target_name']}")


/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [2]:
# 先执行这个 cell 检查将要复制/合并的内容。
api = wandb.Api()
payloads = [build_payload(api, spec) for spec in COPY_SPECS]
for payload in payloads:
    preview_payload(payload)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


target: ft_ir101_s3_full_h-0.333
source runs: ['ft_ir101_s3_full_03-30_0', 'ft_ir101_s3_full_03-31_0']
source ids: ['km5m1xfq', '5n6od08i']
history rows: 1266
history columns: ['val/cplfw/acc', 'train/grad_norm_classifier', 'summary/work_0320_glint_tpir_at_far_1e-07', 'n_images_seen', 'trainer/global_step', 'train/grad_norm_backbone', 'train/loss', 'val/work_0320_3t/tpir_at_far_1e-06', 'step', 'summary/tinyface_rank-1', 'summary/work_0320_3t_tpir_at_far_1e-10', 'val/tinyface/rank-20', 'val/work_0320_3t/tpir_at_far_1e-07', 'val/agedb_30/acc', 'is_best', 'val/tinyface/rank-1', 'val/tinyface/rank-5', 'summary/work_0320_3t_tpir_at_far_1e-08', 'val/agedb_30/std', 'train/mean_loss', 'val/work_0320_glint/tpir_at_far_1e-06', 'summary/calfw_acc', 'summary/work_0320_tpir_at_far_1e-07', 'val/work_0320_3t/tpir_at_far_1e-10', 'summary/work_0320_tpir_at_far_1e-06', 'summary/work_0320_tpir_at_far_1e-09', 'val/cplfw/std', 'summary/work_0320_3t_tpir_at_far_1e-07', 'train/update_ratio_classifier', 'summ

In [3]:
# 确认 preview 没问题后，再执行这个 cell。
api = wandb.Api()
payloads = [build_payload(api, spec) for spec in COPY_SPECS]
for payload in payloads:
    create_target_run(api, payload)



wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


created: e9gfyiqc | https://wandb.ai/kejian-zhao-tsinghua-university/work_0320/runs/e9gfyiqc
step metric = trainer/global_step


uploading:   0%|          | 0/1266 [00:00<?, ?it/s]

epoch,▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
is_best,██▁██████▁
n_images_seen,▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
summary/agedb_30_acc,▄▁▄▄▂▂▆▃▃█
summary/calfw_acc,▇█▄▅▃▅▂▁▄▃
summary/cplfw_acc,▃▄▆▁▆▅▇▅▆█
summary/tinyface_rank-1,▁▂▆▆▆▆▄▇█▇
summary/tinyface_rank-5,▁▂▃▃▆▆▇▇██
summary/work_0320_3t_tpir_at_far_1e-06,▁▂▃▄▄▆▇███
+44,...


finished: ft_ir101_s3_full_h-0.333
